In [ ]:

# @title ۱. نصب پیش‌نیازها و انتخاب مدل Whisper { display-mode: "form" }
import torch

print("⏳ در حال نصب ابزارها (whisper و yt-dlp)...")
!pip install -q faster-whisper yt-dlp gdown

from faster_whisper import WhisperModel

# منوی انتخابی برای کاربر
selected_model = "large-v3-turbo"  # @param ["large-v3-turbo", "large-v3"]

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ سخت‌افزار فعال: {device}")

print(f"⏳ در حال دانلود و بارگذاری مدل {selected_model}...")
model = WhisperModel(selected_model, device=device, compute_type="float16")

print(f"🚀 مدل {selected_model} با موفقیت لود شد و آماده است!")

In [ ]:

# @title ۲. منبع ورودی (لینک یا آپلود مستقیم) { display-mode: "form" }
import glob
import os
import re
from google.colab import files
import yt_dlp

source_type = "آپلود مستقیم از سیستم"  # @param ["لینک (گوگل درایو یا یوتیوب)", "آپلود مستقیم از سیستم"]
media_link = ""  # @param {type:"string"}

# نام پیش‌فرض فایل زیرنویس طبق نظر شما به صورت ثابت
output_srt_name = "subtitle.srt"

input_media_path = "/content/input_file"

# پاکسازی فایل‌های ورودی قبلی برای جلوگیری از تداخل
for f in glob.glob("/content/input_file*"):
    try:
        os.remove(f)
    except Exception:
        pass

if source_type == "آپلود مستقیم از سیستم":
    print("📁 لطفاً فایل صوتی یا ویدیویی خود را انتخاب کنید:")
    uploaded = files.upload()
    if uploaded:
        uploaded_name = list(uploaded.keys())[0]
        ext = os.path.splitext(uploaded_name)[1]
        input_media_path = f"/content/input_file{ext}"
        os.rename(uploaded_name, input_media_path)
        print(f"✅ فایل با موفقیت دریافت و آماده شد: {uploaded_name}")
    else:
        print("❌ هیچ فایلی انتخاب نشد!")

else:
    print("🔍 در حال شناسایی منبع و آماده‌سازی داده‌ها...")

    # حالت اول: لینک یوتیوب (همراه با بای‌پاس ضد ربات کلاینت موبایل)
    if "youtube.com" in media_link or "youtu.be" in media_link:
        ydl_opts = {
            "format": "bestaudio/best",
            "outtmpl": "/content/input_file.%(ext)s",
            "quiet": True,
            "no_warnings": True,
            # شبیه‌سازی درخواست رسمی از اپلیکیشن اندروید/iOS برای دور زدن ربات‌گیر
            "extractor_args": {
                "youtube": {
                    "player_client": ["android", "ios", "mweb"],
                    "player_skip": ["webpage", "configs"],
                }
            },
            "http_headers": {
                "User-Agent": (
                    "com.google.android.youtube/19.09.37 (Linux; U; Android 11;"
                    " en_US)"
                )
            },
        }
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([media_link])

            found_files = glob.glob("/content/input_file.*")
            if found_files:
                input_media_path = found_files[0]
                print("✅ منبع یوتیوب با موفقیت شناسایی و آماده شد.")
            else:
                print("❌ خطا: داده‌ای از یوتیوب استخراج نشد!")
        except Exception as e:
            print(f"❌ خطا در پردازش یوتیوب: {e}")

    # حالت دوم: لینک گوگل درایو
    else:
        import gdown

        match = re.search(r"/file/d/([a-zA-Z0-9_-]+)", media_link) or re.search(
            r"id=([a-zA-Z0-9_-]+)", media_link
        )
        file_id = match.group(1) if match else media_link.strip()

        input_media_path = "/content/input_file.bin"
        gdown.download(id=file_id, output=input_media_path, quiet=True)

        if (
            os.path.exists(input_media_path)
            and os.path.getsize(input_media_path) > 0
        ):
            print("✅ منبع گوگل درایو با موفقیت شناسایی و آماده شد.")
        else:
            print(
                "❌ خطا در دریافت از درایو! مطمئن شوید دسترسی لینک روی 'Anyone"
                " with the link' باشد."
            )

In [ ]:

# @title ۳. اجرای رونویسی و دریافت خروجی { display-mode: "form" }
import ctypes
import datetime
import glob
import os
import subprocess
from google.colab import files
from IPython.display import HTML, display
import soundfile as sf
from tqdm import tqdm


# ۱. حل اتصال CUDA به صورت خودکار
def load_cuda_libraries():
    search_paths = glob.glob(
        "/usr/local/lib/python*/dist-packages/nvidia/*/lib"
    ) + ["/usr/local/cuda/lib64", "/usr/lib/x86_64-linux-gnu"]
    for folder in search_paths:
        if os.path.exists(folder):
            for file in os.listdir(folder):
                if file.startswith("libcublas") or file.startswith("libcudnn"):
                    try:
                        ctypes.CDLL(
                            os.path.join(folder, file), mode=ctypes.RTLD_GLOBAL
                        )
                    except Exception:
                        pass


load_cuda_libraries()

TEMP_AUDIO = "/content/temp_fast_audio.wav"
SILENCE_THRESHOLD = 0.8  # آستانه مکث بین جملات


def format_timestamp(seconds: float) -> str:
    td = datetime.timedelta(seconds=seconds)
    total_seconds = int(td.total_seconds())
    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    millisecs = int((seconds - int(seconds)) * 1000)
    return f"{hours:02d}:{minutes:02d}:{secs:02d},{millisecs:03d}"


# ۲. آماده‌سازی و استانداردسازی صوت (بدون توجه به فرمت اولیه ویدیو یا فایل صوتی)
print("🔊 ۱. آماده‌سازی و بهینه‌سازی فایل صوتی...")
cmd = f'ffmpeg -y -i "{input_media_path}" -vn -acodec pcm_s16le -ar 16000 -ac 1 "{TEMP_AUDIO}" -loglevel error'
subprocess.run(cmd, shell=True, check=True)

# ۳. بارگذاری داده صوتی
audio_data, sample_rate = sf.read(TEMP_AUDIO, dtype="float32")
total_duration = len(audio_data) / sample_rate
print(f"⏱ مدت زمان فایل: {total_duration/60:.1f} دقیقه")

# ۴. آغاز رونویسی با Whisper Turbo
print("🤖 ۲. آغاز رونویسی هوشمند...")
segments_generator, info = model.transcribe(
    audio_data,
    language="en",
    beam_size=5,
    vad_filter=True,
    vad_parameters=dict(min_silence_duration_ms=800),
    word_timestamps=True,
)

all_subtitles = []
current_words = []
cue_start = None

with tqdm(
    total=round(total_duration, 1),
    unit="s",
    desc="⏳ پیشرفت رونویسی",
    dynamic_ncols=True,
) as pbar:
    last_processed_time = 0.0

    for segment in segments_generator:
        progress_increment = max(0.0, segment.end - last_processed_time)
        pbar.update(round(progress_increment, 1))
        last_processed_time = max(last_processed_time, segment.end)
        pbar.set_postfix_str(f"تا دقیقه {segment.end/60:.1f}")

        if not segment.words:
            continue

        for i, word_obj in enumerate(segment.words):
            word_text = word_obj.word.strip()
            if not word_text:
                continue

            w_start = word_obj.start
            w_end = word_obj.end

            if cue_start is None:
                cue_start = w_start

            current_words.append(word_text)

            is_pause = False
            if i < len(segment.words) - 1:
                next_start = segment.words[i + 1].start
                if (next_start - w_end) >= SILENCE_THRESHOLD:
                    is_pause = True
            else:
                is_pause = True

            is_sentence_end = word_text.endswith((".", "?", "!"))

            if is_pause or is_sentence_end:
                all_subtitles.append(
                    {
                        "start": cue_start,
                        "end": w_end,
                        "text": " ".join(current_words),
                    }
                )
                current_words = []
                cue_start = None

    if total_duration > last_processed_time:
        pbar.update(round(total_duration - last_processed_time, 1))

if current_words and cue_start is not None:
    all_subtitles.append(
        {"start": cue_start, "end": w_end, "text": " ".join(current_words)}
    )

# ۵. ساخت فایل SRT
output_srt_path = f"/content/{output_srt_name}"
with open(output_srt_path, "w", encoding="utf-8") as f:
    for idx, sub in enumerate(all_subtitles, 1):
        f.write(f"{idx}\n")
        f.write(
            f"{format_timestamp(sub['start'])} --> {format_timestamp(sub['end'])}\n"
        )
        f.write(f"{sub['text']}\n\n")

# ۶. پاکسازی فایل صوتی موقت
if os.path.exists(TEMP_AUDIO):
    os.remove(TEMP_AUDIO)

print("🎉 پردازش به پایان رسید!")

# ۷. دانلود خودکار زیرنویس در سیستم کاربر
print("⬇️ در حال دانلود خودکار فایل زیرنویس در سیستم شما...")
files.download(output_srt_path)

cta_html = """
<div style="background: linear-gradient(135deg, #0f172a, #1e293b); padding: 25px; border-radius: 16px; text-align: center; max-width: 500px; margin: 25px auto; box-shadow: 0 8px 25px rgba(0,0,0,0.4); font-family: 'Vazirmatn', Tahoma, sans-serif; border: 1px solid #334155;">
    <h3 style="color: #f8fafc; font-size: 17px; margin-bottom: 8px; font-weight: 700;">آموزش‌های بیشتر و ابزارهای هوش مصنوعی</h3>
    <p style="color: #94a3b8; font-size: 14px; margin-bottom: 20px;">برای تماشای آموزش‌های تخصصی و کاربردی، ما را در یوتیوب دنبال کنید:</p>
    <a href="https://www.youtube.com/@aigolden" target="_blank" style="background: linear-gradient(135deg, #ef4444, #dc2626); color: white; padding: 12px 28px; text-decoration: none; font-size: 15px; font-weight: bold; border-radius: 30px; display: inline-flex; align-items: center; gap: 8px; box-shadow: 0 4px 15px rgba(239,68,68,0.4); transition: transform 0.2s;">
        <svg width="22" height="22" viewBox="0 0 24 24" fill="white"><path d="M23.498 6.186a3.016 3.016 0 0 0-2.122-2.136C19.505 3.545 12 3.545 12 3.545s-7.505 0-9.377.505A3.017 3.017 0 0 0 .502 6.186C0 8.07 0 12 0 12s0 3.93.502 5.814a3.016 3.016 0 0 0 2.122 2.136c1.871.505 9.376.505 9.376.505s7.505 0 9.377-.505a3.015 3.015 0 0 0 2.122-2.136C24 15.93 24 12 24 12s0-3.93-.502-5.814zM9.545 15.568V8.432L15.818 12l-6.273 3.568z"/></svg>
        دنبال کردن چنل aigolden
    </a>
</div>
"""
display(HTML(cta_html))